# AI Agent와 외부 API 연동 실습 답안

기상청 **단기예보·중기예보 API**를 LangChain Tool로 감싸고, 농촌진흥청 **농업기술길잡이 PDF 6종**을 검색 Tool로 연결한 Agent입니다.

```text
사용자: "앞으로 10일간 날씨를 분석해서 토마토 농작업 계획을 세워줘."
                │
                ▼
           AI Agent
    ┌───────────┴───────────┐
    ▼                       ▼
단기예보 Tool            중기예보 Tool
(1~3일 상세)             (최대 11일 전망)
    └───────────┬───────────┘
                ▼
        작물 길잡이 검색 Tool  (PDF RAG)
                ▼
          LLM 종합 분석
```

실습 지역은 예제 서울 종로(60, 127)가 아니라, 제공 엑셀에서 직접 찾은 **전주**입니다.


## STEP 0. 환경 준비

`C:\env\.env`에서 키를 읽고 값은 출력하지 않습니다.

- `KMA_SHORT_TERM_KEY` : 단기예보 전용
- `KMA_MID_TERM_KEY` : 중기예보 전용
- `OPENAI_API_KEY` : LLM·임베딩


In [ ]:
import json
import os
import re
import time
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import unquote
from zoneinfo import ZoneInfo

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

warnings.filterwarnings("ignore")

KST = ZoneInfo("Asia/Seoul")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path(r"c:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습\01_AI Agent와 외부 API 연동 실습")
VS_DIR = DATA_DIR / "crop_guide_faiss"

ENV_PATH = r"C:\env\.env"
load_dotenv(ENV_PATH)

KMA_SHORT_TERM_KEY = os.getenv("KMA_SHORT_TERM_KEY")
KMA_MID_TERM_KEY = os.getenv("KMA_MID_TERM_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

missing = [
    name
    for name, value in [
        ("KMA_SHORT_TERM_KEY", KMA_SHORT_TERM_KEY),
        ("KMA_MID_TERM_KEY", KMA_MID_TERM_KEY),
        ("OPENAI_API_KEY", OPENAI_API_KEY),
    ]
    if not value
]
if missing:
    raise ValueError(f"다음 키를 {ENV_PATH}에서 찾을 수 없습니다: {', '.join(missing)}")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", chunk_size=40)

print("KMA_SHORT_TERM_KEY 로드 완료 (키 값은 출력하지 않습니다)")
print("KMA_MID_TERM_KEY 로드 완료 (키 값은 출력하지 않습니다)")
print("OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)")
print("LLM: gpt-4o-mini / Embedding: text-embedding-3-small")


## STEP 1. 조회 지점 확정 (전주)

단기예보는 **격자 nx, ny**, 중기예보는 **구역코드 regId**를 씁니다. 값이 서로 다릅니다.

제공 엑셀에서 전주를 찾아 표로 남깁니다.


In [ ]:
grid_xlsx = next(DATA_DIR.glob("*격자_위경도*.xlsx"))
mid_xlsx = next(DATA_DIR.glob("*중기기온예보구역코드*.xlsx"))

grid_df = pd.read_excel(grid_xlsx)
mid_df = pd.read_excel(mid_xlsx)

print("격자 엑셀:", grid_xlsx.name)
print("중기 엑셀:", mid_xlsx.name)

jeonju_grid = grid_df[
    grid_df["2단계"].astype(str).str.contains("전주시", na=False)
    & grid_df["3단계"].isna()
][["1단계", "2단계", "격자 X", "격자 Y"]]
display(jeonju_grid)

jeonju_mid = mid_df[mid_df["구역명"].astype(str).str.contains("전주|전북", na=False)]
display(jeonju_mid)


In [ ]:
# 전주시완산구·덕진구 대표 격자 (엑셀에서 확인). 서울 종로 60,127 을 쓰지 않음.
LOCATION_NAME = "전주"
NX, NY = 63, 89
LAND_REG_ID = "11F10000"  # 전북자치도 육상예보 (특성 A)
TA_REG_ID = "11F10201"    # 전주 기온예보 (특성 C)
STN_ID = "146"            # 전주 중기전망 지점번호 (중기예보 가이드)

point_df = pd.DataFrame(
    [
        {"항목": "실습 지역", "값": "전주 (전북특별자치도 전주시완산구)", "출처": "격자 엑셀 2단계=전주시완산구"},
        {"항목": "단기예보 격자 nx", "값": NX, "출처": "격자 엑셀 '격자 X'"},
        {"항목": "단기예보 격자 ny", "값": NY, "출처": "격자 엑셀 '격자 Y'"},
        {"항목": "중기육상예보 regId", "값": LAND_REG_ID, "출처": "중기 엑셀 구역명=전북자치도, 특성=A"},
        {"항목": "중기기온예보 regId", "값": TA_REG_ID, "출처": "중기 엑셀 구역명=전주, 특성=C"},
        {"항목": "중기전망 stnId", "값": STN_ID, "출처": "중기예보 활용가이드 지점번호(전주)"},
    ]
)
display(point_df)

print("확인: 육상 regId(광역)와 기온 regId(도시)가 다릅니다.")


## STEP 2. 단기예보 Tool

`KMA_SHORT_TERM_KEY`만 사용합니다. `base_date` / `base_time`은 Asia/Seoul 현재 시각으로 자동 계산합니다.


In [ ]:
SHORT_TERM_BASE = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0"
MID_TERM_BASE = "https://apis.data.go.kr/1360000/MidFcstInfoService"

SKY_CODE = {"1": "맑음", "3": "구름많음", "4": "흐림"}
PTY_CODE = {
    "0": "없음",
    "1": "비",
    "2": "비/눈",
    "3": "눈",
    "4": "소나기",
    "5": "빗방울",
    "6": "빗방울눈날림",
    "7": "눈날림",
}


def call_kma_api(url: str, service_key: str, extra_params: dict) -> dict:
    '''공공데이터포털 JSON 호출. 인증키는 unquote 후 requests가 인코딩합니다.'''
    params = {
        "serviceKey": unquote(service_key),
        "pageNo": "1",
        "numOfRows": "1000",
        "dataType": "JSON",
        **extra_params,
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    try:
        payload = response.json()
    except ValueError as exc:
        preview = response.text[:200].replace(unquote(service_key), "***")
        raise ValueError(f"JSON이 아닌 응답입니다. 미리보기: {preview}") from exc

    header = payload.get("response", {}).get("header", {})
    result_code = header.get("resultCode")
    if result_code != "00":
        raise RuntimeError(f"기상청 API 오류: [{result_code}] {header.get('resultMsg')}")
    return payload["response"]


def extract_items(response_body: dict) -> list[dict]:
    items = response_body.get("body", {}).get("items", {}).get("item", [])
    if isinstance(items, dict):
        return [items]
    return items or []


def latest_vilage_base(now: datetime | None = None) -> tuple[str, str]:
    '''단기예보 최신 발표: 02/05/08/11/14/17/20/23시, 발표 10분 이후.'''
    now = now or datetime.now(KST)
    hours = [2, 5, 8, 11, 14, 17, 20, 23]
    candidates = []
    for hour in hours:
        announced = now.replace(hour=hour, minute=10, second=0, microsecond=0)
        candidates.append((announced, f"{hour:02d}00"))
    available = [(t, bt) for t, bt in candidates if now >= t]
    if available:
        base_dt, base_time = available[-1]
        return base_dt.strftime("%Y%m%d"), base_time
    yesterday = now - timedelta(days=1)
    return yesterday.strftime("%Y%m%d"), "2300"


def latest_ncst_base(now: datetime | None = None) -> tuple[str, str]:
    '''초단기실황: 매시 정시, 약 10분 이후.'''
    now = now or datetime.now(KST)
    base = now.replace(minute=0, second=0, microsecond=0)
    if now.minute < 10:
        base -= timedelta(hours=1)
    return base.strftime("%Y%m%d"), base.strftime("%H00")


def _to_float(value) -> float | None:
    try:
        return float(str(value).replace("mm", "").strip())
    except (TypeError, ValueError):
        return None


def summarize_short_term(nx: int, ny: int) -> str:
    vilage_date, vilage_time = latest_vilage_base()
    ncst_date, ncst_time = latest_ncst_base()

    ncst_items = extract_items(
        call_kma_api(
            f"{SHORT_TERM_BASE}/getUltraSrtNcst",
            KMA_SHORT_TERM_KEY,
            {"base_date": ncst_date, "base_time": ncst_time, "nx": nx, "ny": ny},
        )
    )
    now_obs = {}
    for item in ncst_items:
        cat = item.get("category")
        val = item.get("obsrValue")
        if cat == "T1H":
            now_obs["기온(℃)"] = val
        elif cat == "REH":
            now_obs["습도(%)"] = val
        elif cat == "WSD":
            now_obs["풍속(m/s)"] = val
        elif cat == "RN1":
            now_obs["1시간강수"] = val
        elif cat == "PTY":
            now_obs["강수형태"] = PTY_CODE.get(str(val), val)

    fcst_items = extract_items(
        call_kma_api(
            f"{SHORT_TERM_BASE}/getVilageFcst",
            KMA_SHORT_TERM_KEY,
            {"base_date": vilage_date, "base_time": vilage_time, "nx": nx, "ny": ny},
        )
    )
    by_slot: dict[tuple[str, str], dict] = {}
    for item in fcst_items:
        key = (str(item.get("fcstDate")), str(item.get("fcstTime")))
        slot = by_slot.setdefault(key, {"date": key[0], "time": key[1]})
        cat = item.get("category")
        val = item.get("fcstValue")
        if cat == "TMP":
            slot["tmp"] = _to_float(val)
        elif cat == "POP":
            slot["pop"] = _to_float(val)
        elif cat == "REH":
            slot["reh"] = _to_float(val)
        elif cat == "SKY":
            slot["sky"] = SKY_CODE.get(str(val), val)
        elif cat == "PTY":
            slot["pty"] = PTY_CODE.get(str(val), val)
        elif cat == "PCP":
            slot["pcp"] = val
        elif cat == "WSD":
            slot["wsd"] = _to_float(val)

    daily: dict[str, dict] = {}
    hourly = []
    for (date, time), slot in sorted(by_slot.items()):
        d = daily.setdefault(
            date,
            {"date": f"{date[:4]}-{date[4:6]}-{date[6:]}", "tmps": [], "pops": [], "rehs": [], "skies": [], "ptys": []},
        )
        if slot.get("tmp") is not None:
            d["tmps"].append(slot["tmp"])
        if slot.get("pop") is not None:
            d["pops"].append(slot["pop"])
        if slot.get("reh") is not None:
            d["rehs"].append(slot["reh"])
        if slot.get("sky"):
            d["skies"].append(slot["sky"])
        if slot.get("pty") and slot["pty"] != "없음":
            d["ptys"].append(slot["pty"])
        hh = int(time[:2])
        if hh % 3 == 0:
            hourly.append(
                {
                    "시각": f"{date[:4]}-{date[4:6]}-{date[6:]} {time[:2]}:00",
                    "기온(℃)": slot.get("tmp"),
                    "강수확률(%)": slot.get("pop"),
                    "하늘": slot.get("sky"),
                    "강수형태": slot.get("pty"),
                    "습도(%)": slot.get("reh"),
                }
            )

    daily_rows = []
    for date, d in sorted(daily.items()):
        daily_rows.append(
            {
                "날짜": d["date"],
                "최저기온(℃)": min(d["tmps"]) if d["tmps"] else None,
                "최고기온(℃)": max(d["tmps"]) if d["tmps"] else None,
                "최대강수확률(%)": max(d["pops"]) if d["pops"] else None,
                "평균습도(%)": round(sum(d["rehs"]) / len(d["rehs"]), 1) if d["rehs"] else None,
                "하늘상태": max(set(d["skies"]), key=d["skies"].count) if d["skies"] else None,
                "강수형태": ", ".join(sorted(set(d["ptys"]))) if d["ptys"] else "없음",
            }
        )

    payload = {
        "지역": LOCATION_NAME,
        "자료": "기상청 단기예보 조회서비스 (실황+단기예보)",
        "실황기준": f"{ncst_date} {ncst_time}",
        "예보발표": f"{vilage_date} {vilage_time}",
        "현재실황": now_obs,
        "일별요약": daily_rows,
        "시간별(3시간)": hourly[:24],
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


In [ ]:
@tool
def get_short_term_weather(nx: int = NX, ny: int = NY) -> str:
    '''전주 등 격자의 오늘~글피(약 1~3일) 상세 날씨를 조회합니다.
    기온, 강수확률, 하늘상태, 강수형태, 습도를 요약합니다.
    4일 이후 전망에는 이 도구를 쓰지 말고 중기예보 도구를 쓰세요.
    인증키는 도구 내부에서만 사용하며 사용자에게 출력하지 마세요.

    Args:
        nx: 단기예보 격자 X. 전주는 기본값 63.
        ny: 단기예보 격자 Y. 전주는 기본값 89.
    '''
    try:
        return summarize_short_term(nx, ny)
    except Exception as exc:
        return f"단기예보 조회 실패: {type(exc).__name__}: {exc}"


print("단기예보 Tool 정의 완료:", get_short_term_weather.name)
print("단기 발표 시각:", latest_vilage_base(), "/ 실황:", latest_ncst_base())


## STEP 3. 중기예보 Tool

`KMA_MID_TERM_KEY`만 사용합니다. **육상 regId와 기온 regId는 다릅니다.**
`tmFc`는 06시/18시 중 최신 발표시각으로 자동 계산합니다.


In [ ]:
def latest_mid_tmfc(now: datetime | None = None) -> str:
    '''중기예보 최신 발표시각 YYYYMMDDHHMM. 06시/18시, 발표 10분 이후.'''
    now = now or datetime.now(KST)
    today_06 = now.replace(hour=6, minute=10, second=0, microsecond=0)
    today_18 = now.replace(hour=18, minute=10, second=0, microsecond=0)
    if now >= today_18:
        return now.strftime("%Y%m%d") + "1800"
    if now >= today_06:
        return now.strftime("%Y%m%d") + "0600"
    yesterday = now - timedelta(days=1)
    return yesterday.strftime("%Y%m%d") + "1800"


LAND_TO_STN = {
    "11B00000": "109",
    "11F10000": "146",
    "11F20000": "156",
    "11C20000": "133",
    "11C10000": "131",
    "11H10000": "143",
    "11H20000": "159",
    "11G00000": "184",
    "11D10000": "105",
}


def summarize_mid_term(land_reg_id: str, ta_reg_id: str) -> str:
    tm_fc = latest_mid_tmfc()
    announce_dt = datetime.strptime(tm_fc, "%Y%m%d%H%M").replace(tzinfo=KST)

    outlook = ""
    stn_id = LAND_TO_STN.get(land_reg_id, STN_ID)
    try:
        mid_items = extract_items(
            call_kma_api(
                f"{MID_TERM_BASE}/getMidFcst",
                KMA_MID_TERM_KEY,
                {"stnId": stn_id, "tmFc": tm_fc},
            )
        )
        if mid_items:
            outlook = mid_items[0].get("wfSv", "") or ""
    except Exception:
        outlook = "(중기전망 텍스트는 이 시각에 제공되지 않았습니다)"

    land_item = extract_items(
        call_kma_api(
            f"{MID_TERM_BASE}/getMidLandFcst",
            KMA_MID_TERM_KEY,
            {"regId": land_reg_id, "tmFc": tm_fc},
        )
    )[0]
    ta_item = extract_items(
        call_kma_api(
            f"{MID_TERM_BASE}/getMidTa",
            KMA_MID_TERM_KEY,
            {"regId": ta_reg_id, "tmFc": tm_fc},
        )
    )[0]

    days = []
    for day in range(3, 11):
        forecast_date = (announce_dt + timedelta(days=day)).strftime("%Y-%m-%d")
        if day <= 7:
            wf_am, wf_pm = land_item.get(f"wf{day}Am"), land_item.get(f"wf{day}Pm")
            pop_am, pop_pm = land_item.get(f"rnSt{day}Am"), land_item.get(f"rnSt{day}Pm")
        else:
            wf_am = wf_pm = land_item.get(f"wf{day}")
            pop_am = pop_pm = land_item.get(f"rnSt{day}")
        days.append(
            {
                "일차": f"+{day}일",
                "날짜": forecast_date,
                "날씨(오전)": wf_am,
                "날씨(오후)": wf_pm,
                "강수확률(오전)%": pop_am,
                "강수확률(오후)%": pop_pm,
                "최저기온(℃)": ta_item.get(f"taMin{day}"),
                "최고기온(℃)": ta_item.get(f"taMax{day}"),
            }
        )

    payload = {
        "지역": LOCATION_NAME,
        "자료": "기상청 중기예보 조회서비스 (육상+기온+전망)",
        "발표시각": tm_fc,
        "안내": "육상예보 구역과 기온예보 구역은 서로 다른 코드입니다. 1~3일 상세는 단기예보를 참고하세요.",
        "중기전망": outlook,
        "일별전망": days,
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


In [ ]:
@tool
def get_mid_term_weather(land_reg_id: str = LAND_REG_ID, ta_reg_id: str = TA_REG_ID) -> str:
    '''중기예보로 +3일~+10일(최대 11일) 날씨·강수확률·최저기온·최고기온을 조회합니다.
    육상예보 구역코드(land_reg_id)와 기온예보 구역코드(ta_reg_id)는 서로 다릅니다.
    전주 기본값: 육상=전북자치도, 기온=전주.
    앞으로 일주일/10일 전망, 4일 이후 날씨에 사용하세요.
    오늘~글피 시간별 상세는 단기예보 도구를 쓰세요.

    Args:
        land_reg_id: 중기육상예보 광역 구역코드. 전북은 11F10000.
        ta_reg_id: 중기기온예보 도시 구역코드. 전주는 11F10201.
    '''
    try:
        return summarize_mid_term(land_reg_id, ta_reg_id)
    except Exception as exc:
        return f"중기예보 조회 실패: {type(exc).__name__}: {exc}"


print("중기예보 Tool 정의 완료:", get_mid_term_weather.name)
print("중기 발표시각 tmFc:", latest_mid_tmfc())
print("육상 regId / 기온 regId 가 다름:", LAND_REG_ID, "vs", TA_REG_ID)


## STEP 4. 작물 길잡이 검색 Tool (PDF RAG)

농업기술길잡이 PDF 6종을 청킹·임베딩하고 FAISS에 persist 합니다. 재실행 시 재임베딩하지 않습니다.

메타데이터: `crop`, `source`, `page`


In [ ]:
ALLOWED_CROPS = ("토마토", "고추", "수박", "참외", "고구마", "사과")
CROP_KEYWORDS = [
    ("토마토", "토마토"),
    ("고추", "고추"),
    ("수박", "수박"),
    ("참외", "참외"),
    ("고구마", "고구마"),
    ("사과", "사과"),
]


def crop_from_filename(name: str) -> str | None:
    for keyword, crop in CROP_KEYWORDS:
        if keyword in name:
            return crop
    return None


pdf_files = [
    path
    for path in DATA_DIR.iterdir()
    if path.suffix.lower() == ".pdf" and crop_from_filename(path.name)
]
pdf_files = sorted(pdf_files, key=lambda p: crop_from_filename(p.name) or p.name)

pdf_map_df = pd.DataFrame(
    [
        {"작물": crop_from_filename(path.name), "파일명": path.name, "크기(MB)": round(path.stat().st_size / 1_000_000, 1)}
        for path in pdf_files
    ]
)
display(pdf_map_df)
print("PDF 수:", len(pdf_files), "/ 기대: 6")


In [ ]:
def load_pdf_documents(pdf_paths: list[Path]) -> list[Document]:
    docs: list[Document] = []
    for path in pdf_paths:
        crop = crop_from_filename(path.name)
        reader = PdfReader(str(path))
        kept = 0
        for idx, page in enumerate(reader.pages, start=1):
            text = (page.extract_text() or "").strip()
            text = re.sub(r"[ \t]+", " ", text)
            text = re.sub(r"\n{3,}", "\n\n", text)
            if len(text) < 80:
                continue
            docs.append(
                Document(
                    page_content=text,
                    metadata={"crop": crop, "source": path.name, "page": idx},
                )
            )
            kept += 1
        print(f"  {crop}: {path.name} → {len(reader.pages)}쪽 중 {kept}쪽 사용")
    return docs


index_file = VS_DIR / "index.faiss"
if index_file.exists():
    print("기존 FAISS 인덱스를 로드합니다. 재임베딩하지 않습니다.")
    vectorstore = FAISS.load_local(
        str(VS_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print("로드된 벡터 수:", vectorstore.index.ntotal)
else:
    print("PDF를 로드하고 벡터스토어를 새로 만듭니다. (최초 1회)")
    raw_docs = load_pdf_documents(pdf_files)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(raw_docs)
    print("원본 페이지 문서:", len(raw_docs), "/ 청크:", len(chunks))

    def add_batch(store, batch, start_idx: int):
        for attempt in range(8):
            try:
                if store is None:
                    return FAISS.from_documents(batch, embeddings)
                store.add_documents(batch)
                return store
            except Exception as exc:
                msg = str(exc).lower()
                if "rate" in msg or "429" in msg:
                    wait = 20 + attempt * 10
                    print(f"  rate limit at {start_idx}, {wait}초 대기 (시도 {attempt + 1}/8)")
                    time.sleep(wait)
                    continue
                raise
        raise RuntimeError("임베딩 rate limit을 여러 번 재시도했지만 실패했습니다.")

    vectorstore = None
    batch_size = 40
    for start in range(0, len(chunks), batch_size):
        batch = chunks[start : start + batch_size]
        print(f"  임베딩 {start + 1}~{start + len(batch)} / {len(chunks)}")
        vectorstore = add_batch(vectorstore, batch, start + 1)
        time.sleep(2)

    VS_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(VS_DIR))
    print("FAISS persist 완료:", VS_DIR)


In [ ]:
def format_crop_hits(docs: list[Document]) -> str:
    if not docs:
        return "검색된 농업기술길잡이 문맥이 없습니다. 없는 내용은 만들어 내지 마세요."
    blocks = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata
        excerpt = doc.page_content.strip().replace("\n", " ")
        if len(excerpt) > 700:
            excerpt = excerpt[:700] + "..."
        blocks.append(
            f"[{i}] 작물={meta.get('crop')} | 파일={meta.get('source')} | 페이지={meta.get('page')}\n{excerpt}"
        )
    return "아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.\n\n" + "\n\n".join(blocks)


@tool
def search_crop_guide(query: str, crop: str = "") -> str:
    '''농촌진흥청 농업기술길잡이 PDF에서 작물 재배 지식을 검색합니다.
    가능한 작물: 토마토, 고추, 수박, 참외, 고구마, 사과.
    목록에 없는 작물(딸기, 벼 등)은 다른 작물 내용으로 대체하지 말고 자료 없음을 반환합니다.
    검색된 발췌에 없는 재배법·약제명·수치를 만들지 마세요.
    날씨 API 대신 이 도구를 재배/육묘/정식/시비/병해충/수확 질문에 사용하세요.

    Args:
        query: 찾고 싶은 재배 내용 (예: 육묘와 정식 시 주의사항).
        crop: 작물명. 토마토/고추/수박/참외/고구마/사과 중 하나. 없으면 빈 문자열.
    '''
    crop = (crop or "").strip()
    if crop and crop not in ALLOWED_CROPS:
        return (
            f"제공된 농업기술길잡이 PDF에 '{crop}' 자료가 없습니다. "
            f"가능한 작물: {', '.join(ALLOWED_CROPS)}. "
            "다른 작물 내용을 해당 작물인 것처럼 사용하지 마세요. "
            "주어진 자료만으로는 알 수 없다고 답하세요."
        )
    try:
        search_kwargs = {"k": 5}
        if crop:
            search_kwargs["filter"] = {"crop": crop}
            search_kwargs["fetch_k"] = 40
        docs = vectorstore.similarity_search(query, **search_kwargs)
        if crop:
            docs = [d for d in docs if d.metadata.get("crop") == crop]
        return format_crop_hits(docs)
    except Exception as exc:
        return f"작물 길잡이 검색 실패: {type(exc).__name__}: {exc}"


print("작물 검색 Tool 정의 완료:", search_crop_guide.name)
sample = search_crop_guide.invoke({"query": "육묘와 정식", "crop": "토마토"})
print("토마토 검색 미리보기:\n")
print(sample[:800])


## STEP 5. Agent 구성

질문의 시간 범위와 작물 여부에 따라 Tool을 선택합니다. 근거 없는 내용은 지어내지 않습니다.


In [ ]:
SYSTEM_PROMPT = """당신은 농업 의사결정을 돕는 Agent입니다.
날씨는 기상청 API Tool 결과만, 재배법은 농업기술길잡이 PDF 검색 결과만 근거로 답합니다.
없는 숫자·약제명·작업법을 추측하지 마세요. 근거가 없으면 '주어진 자료만으로는 알 수 없다'고 말하세요.

[도구 선택]
- 오늘/내일/모레, 1~3일 상세 날씨 → get_short_term_weather
- 4일 이후, 앞으로 일주일/10일 전망 → get_mid_term_weather
- 10일 계획처럼 단기+중기가 모두 필요하면 둘 다 호출
- 재배/병해충/정식/시비/수확/육묘 등 작물 지식 → search_crop_guide
- 작물이 명시되지 않은 단순 날씨 질문에는 PDF 검색을 하지 마세요
- 날씨가 필요 없는 재배 지식 질문에는 기상청 API를 호출하지 마세요

[작물]
- PDF에 있는 작물만 검색: 토마토, 고추, 수박, 참외, 고구마, 사과
- 딸기, 벼 등 PDF에 없는 작물은 다른 작물 내용으로 대체하지 말고 자료 부족을 명시하세요

[답변 형식]
- 날씨 답에는 실제 예보 수치(기온, 강수확률 등)를 포함하세요
- PDF를 썼으면 파일명과 페이지를 근거로 밝히세요
- 인증키, 격자 좌표, 구역코드를 사용자에게 출력하지 마세요
- 실습 기본 지역은 전주입니다. 도구 기본 인자를 그대로 쓰면 됩니다
"""

agent = create_agent(
    model=llm,
    tools=[get_short_term_weather, get_mid_term_weather, search_crop_guide],
    system_prompt=SYSTEM_PROMPT,
)

print("Agent 준비 완료. Tools:", [t.name for t in [get_short_term_weather, get_mid_term_weather, search_crop_guide]])


In [ ]:
def message_text(msg) -> str:
    content = getattr(msg, "content", "")
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("text"):
                parts.append(str(block["text"]))
        return "\n".join(parts)
    return str(content)


def extract_tool_calls(result: dict) -> list[str]:
    names = []
    for msg in result.get("messages", []):
        for call in getattr(msg, "tool_calls", None) or []:
            name = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
            if name:
                names.append(name)
    return names


def run_scenario(label: str, question: str, expected: str) -> dict:
    print("=" * 78)
    print(f"시나리오 {label}")
    print("질문:", question)
    print("기대한 Tool:", expected)
    print("=" * 78)

    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 30},
    )
    called = extract_tool_calls(result)
    final = message_text(result["messages"][-1])

    print("\n[실제 호출 Tool]")
    print(", ".join(called) if called else "(없음)")
    print("\n[도구 호출 과정]")
    for msg in result["messages"]:
        role = type(msg).__name__
        if getattr(msg, "tool_calls", None):
            for call in msg.tool_calls:
                name = call.get("name") if isinstance(call, dict) else getattr(call, "name", "")
                args = call.get("args") if isinstance(call, dict) else getattr(call, "args", {})
                safe_args = {k: v for k, v in (args or {}).items() if "key" not in str(k).lower()}
                print(f"  → {name}({safe_args})")
        elif role == "ToolMessage":
            preview = message_text(msg).replace("\n", " ")[:180]
            print(f"  ← Tool 결과 미리보기: {preview}...")
    print("\n[최종 답변]")
    print(final)
    print()

    record = {
        "시나리오": label,
        "질문": question,
        "기대한 Tool": expected,
        "실제 호출 Tool": ", ".join(called) if called else "(없음)",
        "날씨 근거 여부": "Y" if any(n.startswith("get_") and "weather" in n for n in called) else "N",
        "PDF 근거 여부": "Y" if "search_crop_guide" in called else "N",
        "최종답변": final,
        "호출목록": called,
    }
    return record


SCENARIO_RESULTS: list[dict] = []
print("시나리오 실행 함수 준비 완료")


## STEP 6. 실행 시나리오

같은 Agent로 A~E를 실제로 실행하고, 호출된 Tool과 최종 답을 남깁니다.


### (A) 단기만


In [ ]:
q_a = (
    f"오늘과 내일 {LOCATION_NAME} 날씨를 알려줘. "
    "강수 가능성이 있으면 방제도 함께 주의해야 하는지 짧게 말해줘."
)
SCENARIO_RESULTS.append(
    run_scenario("A", q_a, "get_short_term_weather (PDF는 작물 미명시 시 생략이 바람직)")
)


### (B) 중기만


In [ ]:
q_b = f"앞으로 10일간 {LOCATION_NAME} 날씨 전망과 기온을 정리해줘."
SCENARIO_RESULTS.append(
    run_scenario("B", q_b, "get_mid_term_weather (상세가 필요하면 단기도 가능)")
)


### (C) PDF만


In [ ]:
q_c = "토마토 육묘와 정식 시 주의사항을 농업기술길잡이 기준으로 요약해줘."
SCENARIO_RESULTS.append(
    run_scenario("C", q_c, "search_crop_guide 만 (날씨 API 호출 금지)")
)


### (D) API + PDF 결합 (핵심)


In [ ]:
q_d = f"앞으로 10일간 {LOCATION_NAME} 날씨를 분석해서 토마토 농작업 계획을 세워줘."
SCENARIO_RESULTS.append(
    run_scenario("D", q_d, "get_short_term_weather + get_mid_term_weather + search_crop_guide")
)


### (E) 자료 없음 / 잘못된 라우팅 방지


In [ ]:
q_e = "딸기 재배에서 정식 후 물 관리 방법을 알려줘."
SCENARIO_RESULTS.append(
    run_scenario("E", q_e, "search_crop_guide (자료 부족 명시, 다른 작물로 대체 금지)")
)


## STEP 7. 결과 정리

시나리오 A~E 비교표와 (D) 농작업 일정입니다.


In [ ]:
summary_df = pd.DataFrame(
    [
        {
            "질문": r["질문"],
            "기대한 Tool": r["기대한 Tool"],
            "실제 호출 Tool": r["실제 호출 Tool"],
            "날씨 근거 여부": r["날씨 근거 여부"],
            "PDF 근거 여부": r["PDF 근거 여부"],
            "비고": "",
        }
        for r in SCENARIO_RESULTS
    ],
    index=[r["시나리오"] for r in SCENARIO_RESULTS],
)

notes = {
    "A": "1~3일 상세 → 단기. 작물 미명시 시 PDF 생략이 바람직",
    "B": "10일 전망 → 중기. 단기는 선택",
    "C": "재배 지식만 → PDF. 날씨 API 없어야 함",
    "D": "핵심: 단기+중기+토마토 PDF, 수치와 파일명·페이지 근거",
    "E": "딸기 PDF 없음. 다른 작물로 대체하면 안 됨",
}
summary_df["비고"] = [notes[i] for i in summary_df.index]
display(summary_df)


In [ ]:
d_rec = next(r for r in SCENARIO_RESULTS if r["시나리오"] == "D")
print("시나리오 D 호출 Tool:", d_rec["실제 호출 Tool"])
print("-" * 78)
print("시나리오 D 최종 조언 (농작업 계획)")
print("-" * 78)
print(d_rec["최종답변"])


## 체크리스트

- [x] 키는 `.env`에서만 읽고 값을 출력하지 않음
- [x] 전주 nx/ny, 육상·기온 regId를 엑셀에서 찾아 표로 남김 (서울 종로 좌표 미사용)
- [x] 단기 Tool과 중기 Tool의 인증키·파라미터가 분리됨
- [x] `base_time`, `tmFc`를 현재 시각 기준으로 자동 계산
- [x] PDF 6종 인덱싱, crop/source/page 메타데이터
- [x] FAISS persist로 재실행 시 재임베딩 생략
- [x] 시나리오 A~E 실제 실행, 도구 호출 트레이스와 비교표 보존
